# Milestone 1a: Data Cleaning

## Loading the Data (2013–2025)

In Milestone 0 we chose our three stations (Dublin, Toronto and San Jose), our elements of interest and our time range (2014–2025).

Each yearly file contains tens of millions of rows from stations all over the world, so loading all 12 years at once would use far too much memory. Instead, we load **one year at a time**, and right after each download we keep only the rows for our three stations and our elements of interest. The filtered years are then combined into one dataset.

In [1]:
import pandas as pd

yearly = []
cols = ['STATION', 'DATE', 'ELEMENT', 'VALUE', 'M_FLAG', 'Q_FLAG', 'S_FLAG', 'OBS_TIME']
elements = ['TMAX', 'TMIN', 'TAVG', 'PRCP', 'SNOW', 'AWND']
selected_stations = ['EI000003969', 'CAN06158731', 'USW00023293']

# Note: this will take 20+ minutes
for year in range(2014, 2026):
    url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/" + str(year) + ".csv.gz"

    # dtype as an argumnet to prevent Dtype warnings about mixed types
    weather_x = pd.read_csv(url, header=None, names=cols, dtype={'M_FLAG':str, 'Q_FLAG':str, 'S_FLAG':str})
    print('\n', '-' * 40)
    print(f'{year} download complete.')

    weather_x = weather_x[(weather_x.STATION.isin(selected_stations)) & (weather_x.ELEMENT.isin(elements))]
    print('Filtering complete.\n')

    print(f'** Year {year} **:\nNumber of rows: {weather_x.shape[0]}\nNumber of columns: {weather_x.shape[1]}\n\nELEMENT counts per STATION:\n')
    print(weather_x.groupby(['STATION','ELEMENT']).size())
    yearly.append(weather_x)

weather = pd.concat(yearly, ignore_index=True)



 ----------------------------------------
2014 download complete.
Filtering complete.

** Year 2014 **:
Number of rows: 4745
Number of columns: 8

ELEMENT counts per STATION:

STATION      ELEMENT
CAN06158731  PRCP       365
             SNOW       365
             TAVG       365
             TMAX       365
             TMIN       365
EI000003969  PRCP       365
             TAVG       365
             TMAX       365
             TMIN       365
USW00023293  AWND       365
             PRCP       365
             TMAX       365
             TMIN       365
dtype: int64

 ----------------------------------------
2015 download complete.
Filtering complete.

** Year 2015 **:
Number of rows: 4735
Number of columns: 8

ELEMENT counts per STATION:

STATION      ELEMENT
CAN06158731  PRCP       358
             SNOW       362
             TAVG       365
             TMAX       365
             TMIN       365
EI000003969  PRCP       365
             TAVG       365
             TMAX       365
   

In [3]:
print('**Combined years 2014 - 2025 dataset audit:**')
print('\n', '-'*20, '\n')
weather.info()
print('\n', '-'*20, '\n')
print('Duplicate rows:', weather.duplicated().sum())
print('Failed quality checks (Q_FLAG):', weather.Q_FLAG.notna().sum())
print('\n', '-'*20, '\n')
print('Readings per station and element:')
display(weather.groupby(['STATION', 'ELEMENT']).size().unstack())
print('Data snippet:')
weather.head()

**Combined years 2014 - 2025 dataset audit:**

 -------------------- 

<class 'pandas.DataFrame'>
RangeIndex: 56719 entries, 0 to 56718
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   STATION   56719 non-null  str    
 1   DATE      56719 non-null  int64  
 2   ELEMENT   56719 non-null  str    
 3   VALUE     56719 non-null  int64  
 4   M_FLAG    5522 non-null   str    
 5   Q_FLAG    83 non-null     str    
 6   S_FLAG    56719 non-null  str    
 7   OBS_TIME  13128 non-null  float64
dtypes: float64(1), int64(2), str(5)
memory usage: 3.5 MB

 -------------------- 

Duplicate rows: 0
Failed quality checks (Q_FLAG): 83

 -------------------- 

Readings per station and element:


ELEMENT,AWND,PRCP,SNOW,TAVG,TMAX,TMIN
STATION,,,,,,
CAN06158731,NaN,4364.0,4362.0,4357.0,4359.0,4358.0
EI000003969,NaN,4383.0,NaN,4241.0,4383.0,4383.0
USW00023293,4374.0,4382.0,27.0,NaN,4374.0,4372.0


Data snippet:


,STATION,DATE,ELEMENT,VALUE,M_FLAG,Q_FLAG,S_FLAG,OBS_TIME
0,USW00023293,20140101,TMAX,167,NaN,NaN,W,2400.0
1,USW00023293,20140101,TMIN,17,NaN,NaN,W,2400.0
2,USW00023293,20140101,PRCP,0,NaN,NaN,W,2400.0
3,USW00023293,20140101,AWND,4,NaN,NaN,W,NaN
4,CAN06158731,20140101,TMAX,-100,NaN,NaN,C,NaN


In [4]:
# Cleaning:

# filter out inaccurate reports 
weather = weather[weather.Q_FLAG.isna()]

# drop extra flags
weather = weather.drop(columns=['M_FLAG', 'Q_FLAG', 'S_FLAG', 'OBS_TIME'])

# rename STATION
weather['STATION'] = weather.STATION.map({'EI000003969':'Dublin', 'CAN06158731':'Toronto', 'USW00023293':'San Jose'})


In [5]:
# Reshape to Wide

weather_wide = weather.pivot(index=['STATION', 'DATE'], columns='ELEMENT', values='VALUE').reset_index().rename_axis(columns=None)
weather_wide.head()

,STATION,DATE,AWND,PRCP,SNOW,TAVG,TMAX,TMIN
0,Dublin,20140101,NaN,38.0,NaN,63.0,105.0,27.0
1,Dublin,20140102,NaN,37.0,NaN,68.0,103.0,47.0
2,Dublin,20140103,NaN,31.0,NaN,68.0,83.0,36.0
3,Dublin,20140104,NaN,0.0,NaN,39.0,72.0,12.0
4,Dublin,20140105,NaN,44.0,NaN,69.0,128.0,12.0


In the wide table, NaNs mean that the station didn't record this element (for example, Dublin does not report SNOW), or that a reading is missing for that day. We leave them as NaN and will not use any imputation strategy, as the NaN is meaningful here.

In [6]:
# Cleaning:

# Convert columns from 'tenths of degrees' and 'tenths of mm'
weather_wide[['TMAX', 'TMIN', 'TAVG', 'PRCP', 'AWND']] = weather_wide[['TMAX', 'TMIN', 'TAVG', 'PRCP', 'AWND']] / 10

# Convert to datetime
weather_wide['DATE'] = pd.to_datetime(weather_wide['DATE'], format='%Y%m%d')
# USE pd.read_csv(..., parse_dates=['DATE']) when re-reading from data/ to get the datetime

In [7]:
# Add columns

season_dic = {
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Spring',  4:'Spring', 5:'Spring', 
    6:'Summer', 7:'Summer', 8:'Summer', 
    9:'Fall', 10:'Fall', 11:'Fall', 
}
weather_wide['SEASON'] = weather_wide.DATE.dt.month.map(season_dic)

**Cleaning / Reshaping decisions:**
1. **Removed readings that failed quality checks:** 89 rows had a non-null Q_FLAG, meaning they failed one of NOAA's quality checks and may be unreliable, so we removed them.
2. **Dropped the flag columns:** M_FLAG, Q_FLAG, S_FLAG and OBS_TIME are extra notes we don't need for our analysis (Q_FLAG is no longer needed once the failed readings are removed).
3. **Renamed station IDs to city names:** IDs like `EI000003969` are hard to read and remember, so we replaced them with Dublin, Toronto and San Jose. This makes our tables, plots and legends easier to understand without a lookup table. The full station details are kept in a separate metadata file.
4. **Reshaped from long to wide:** The raw data has one row per measurement (station, date, element, value). We pivoted it so each row is one station on one day, with a column for each element (TMAX, TMIN, PRCP, etc.). This makes it much easier to compare and plot measurements.
5. **Converted units:** NOAA stores TMAX, TMIN, TAVG, PRCP and AWND in tenths (for example, 100 means 10.0 °C), so we divided by 10.
6. **Converted DATE to datetime:** DATE was stored as a number like 20130101, so we changed it to a proper date. This lets us sort by date, pull out the month or year, and plot over time.
7. **Added a SEASON column:** Our data is almost all numbers, with very few categorical columns to group by. We added a SEASON column (Winter, Spring, Summer, Fall), so we can aggregate by season when plotting. This helps if daily data causes overplotting.

In [8]:
# load stations 

url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt"
colwidths = [(0,11), (12,20), (21,30), (31,37), (38,40), (41,71), (72,75), (76,79), (80,85)]
cols = ['ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'STATE', 'NAME', 'GSN FLAG', 'HCN/CRN FLAG', 'WMO ID']

stations = pd.read_fwf(url, colspecs=colwidths, header=None, names=cols)

# we only need information on three
stations = stations[stations.ID.isin(selected_stations)]

# keep relavant columns
stations = stations[['ID', 'NAME', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'STATE']]

# Add key for an easy merge
stations['STATION'] = stations.ID.map({'EI000003969': 'Dublin', 'CAN06158731': 'Toronto', 'USW00023293': 'San Jose'})

# export to data/
stations.to_csv('../data/stations.csv', index=False)

stations

,ID,NAME,LATITUDE,LONGITUDE,ELEVATION,STATE,STATION
31779,CAN06158731,TORONTO INTL A,43.6767,-79.6306,173.4,ON,Toronto
34121,EI000003969,DUBLIN PHOENIX PARK,53.3639,-6.3192,49.0,NaN,Dublin
131289,USW00023293,SAN JOSE,37.3594,-121.9244,14.9,CA,San Jose


**Save station metadata:** we keep the name, location and elevation for our three stations in a separate file, in case we want to use them in our analysis. The STATION column matches the city names in our weather data, so the two files can be joined when needed.

In [9]:
# final 2014 - 2025, 3 station dataset

# shuffle to get a better snapshot (since its grouped by station and ordered by date)
weather_wide.sample(frac=1).head()

,STATION,DATE,AWND,PRCP,SNOW,TAVG,TMAX,TMIN,SEASON
7839,San Jose,2023-06-20,3.4,0.0,NaN,NaN,23.3,10.0,Summer
11932,Toronto,2022-09-08,NaN,0.0,0.0,20.9,26.7,15.0,Fall
195,Dublin,2014-07-15,NaN,0.2,NaN,15.5,20.8,12.3,Summer
1011,Dublin,2016-10-08,NaN,0.0,NaN,12.9,15.3,10.7,Fall
7011,San Jose,2021-03-13,2.7,0.0,NaN,NaN,15.0,5.6,Spring


**Cleaned dataset columns:**
- **STATION**: City name of the weather station (Dublin, Toronto or San Jose).
- **DATE**: Date of the reading (datetime).
- **AWND**: Average daily wind speed, in m/s (San Jose only).
- **PRCP**: Precipitation (rain plus melted snow), in mm.
- **SNOW**: Snowfall that day, in mm (Toronto, plus a few San Jose readings).
- **TAVG**: Daily average temperature, in °C (Dublin and Toronto only).
- **TMAX**: Daily maximum temperature, in °C.
- **TMIN**: Daily minimum temperature, in °C.
- **SEASON**: Season based on the month (Winter = Dec to Feb, Spring = Mar to May, Summer = Jun to Aug, Fall = Sep to Nov).

In [10]:
# save to data/ folder 

weather_wide.to_csv('../data/weather_14-25.csv', index=False)